# ⚙️ CORE PIPELINE: HỆ THỐNG SỐ HÓA TỦ SÁCH TOÀN DIỆN
**Thực hiện:** Nguyễn Tùng Lâm

Notebook này trình bày quy trình kỹ thuật bóc tách dữ liệu từ ảnh chụp gáy sách sử dụng mô hình YOLOv5x6 và TransformerOCR.

## A. Cấu hình Hệ thống & Môi trường

In [ ]:
# 1. DỌN DẸP VÀ CLONE MÃ NGUỒN
import os, shutil
%cd /content/
if os.path.exists('bookcase-digitization'): shutil.rmtree('bookcase-digitization')
!git clone https://github.com/pie-12/bookcase-digitization.git
%cd bookcase-digitization

# 2. CÀI ĐẶT THƯ VIỆN & VÁ LỖI TƯƠNG THÍCH
print("🛠 Đang cấu hình hệ thống (Numpy fix, OpenCV headless)...")
!pip install "numpy<2" opencv-python-headless==4.8.0.74 --force-reinstall -q
!pip install craft-text-detector vietocr==0.3.5 --no-deps -q
!pip install albumentations==1.4.2 einops gdown prefetch-generator shapely scikit-image -q
!git clone https://github.com/ultralytics/yolov5 -q

print("🩹 Đang vá lỗi mã nguồn thư viện...")
vgg_path = "/usr/local/lib/python3.12/dist-packages/craft_text_detector/models/basenet/vgg16_bn.py"
if os.path.exists(vgg_path):
    with open(vgg_path, 'r') as f: content = f.read()
    with open(vgg_path, 'w') as f: f.write(content.replace("from torchvision.models.vgg import model_urls", "model_urls = {}"))

vocr_path = "/usr/local/lib/python3.12/dist-packages/vietocr/tool/translate.py"
if os.path.exists(vocr_path):
    with open(vocr_path, 'r') as f: content = f.read()
    with open(vocr_path, 'w') as f: f.write(content.replace("Image.ANTIALIAS", "Image.Resampling.LANCZOS"))

print("✅ Môi trường đã sẵn sàng.")

## B. Tiền xử lý Hình ảnh (Pre-processing)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import Utlis as utlis

img_path = '/content/bookcase-digitization/data_test/1624445642850.jpg'
img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 1. Ảnh gốc
plt.figure(figsize=(15, 6))
plt.subplot(1, 3, 1); plt.title("1. Ảnh thô từ Camera"); plt.imshow(img_rgb); plt.axis('off')

# 2. Thuật toán Canny Edge Detection
heightImg, widthImg = 720, 540
img_resize = cv2.resize(img, None, fx=0.3, fy=0.3)
imgGray = cv2.cvtColor(img_resize, cv2.COLOR_BGR2GRAY)
imgBlur = cv2.GaussianBlur(imgGray, (5, 5), 0)
imgThreshold = cv2.Canny(imgBlur, 30, 50)
kernel = np.ones((5, 5))
imgDial = cv2.dilate(imgThreshold, kernel, iterations=2)
imgThreshold = cv2.erode(imgDial, kernel, iterations=1)
plt.subplot(1, 3, 2); plt.title("2. Canny Edge & Contour"); plt.imshow(imgThreshold, cmap='gray'); plt.axis('off')

# 3. Perspective Transform (Bẻ phẳng)
contours, _ = cv2.findContours(imgThreshold, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
biggest, maxArea = utlis.biggestContour(contours)
if biggest.size != 0 and maxArea > 5000:
    biggest = utlis.reorder(biggest)
    pts1 = np.float32(biggest)
    pts2 = np.float32([[0, 0], [widthImg, 0], [0, heightImg], [widthImg, heightImg]])
    matrix = cv2.getPerspectiveTransform(pts1, pts2)
    imgWarp = cv2.warpPerspective(img_resize, matrix, (widthImg, heightImg))
    plt.subplot(1, 3, 3); plt.title("3. Perspective Transform"); plt.imshow(cv2.cvtColor(imgWarp, cv2.COLOR_BGR2RGB)); plt.axis('off')
else:
    print("⚠️ Không tìm thấy khung gáy sách!")
plt.show()

## C. Suy luận & Trực quan hóa YOLOv5

In [ ]:
# Chạy nhận diện mô phỏng để lấy kết quả hình ảnh chuẩn
!python run_presentation.py

In [ ]:
from IPython.display import Image, display
import os
detect_file = '/content/bookcase-digitization/runs/detect/detected_1624445642850.jpg'
if os.path.exists(detect_file):
    print("✅ Kết quả nhận diện vùng thông tin (YOLOv5x6):")
    display(Image(filename=detect_file, width=600))
else:
    print("❌ Không tìm thấy ảnh kết quả. Hãy kiểm tra lại Step C.4.")

## D. Kết quả trích xuất văn bản (OCR)

In [ ]:
import pandas as pd
csv_path = '/content/bookcase-digitization/ket_qua_thuyet_trinh.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("✅ Bảng dữ liệu trích xuất thành công:")
    display(df[['file names', 'Ten sach', 'Tac gia', 'Nha xuat ban', 'Tap']].head(10))
else:
    print("❌ Không tìm thấy file CSV kết quả.")